<a href="https://www.kaggle.com/code/tommasofacchin/01-resident-evil-data-scraping-and-generation?scriptVersionId=305026291" target="_blank"><img align="left" alt="Kaggle" title="Open in Kaggle" src="https://kaggle.com/static/images/open-in-kaggle.svg"></a>

In [1]:
import numpy as np 
import pandas as pd 
import requests
from pathlib import Path
from bs4 import BeautifulSoup

# Datesets

**Game** (id, title, year, type)

**Character** (id, name, role, faction)

**GameAppearence** (gameId, characterId, role)

**Dialogues** (id, game_id, scene_id, line_index, character_id, text, timestamp, source)

**Scene** (id, game_id, name, sequence_index)

## Games

In [2]:
games_data = [
    # id, title, year, type, chronology_order
    (1,  "Resident Evil",                             1996, "mainline", 2),
    (2,  "Resident Evil 2",                           1998, "mainline", 4),
    (3,  "Resident Evil 3: Nemesis",                  1999, "mainline", 3),
    (4,  "Resident Evil: Code – Veronica",            2000, "mainline", 5),
    (5,  "Resident Evil 0",                           2002, "mainline", 1),
    (6,  "Resident Evil 4",                           2005, "mainline", 6),
    (7,  "Resident Evil 5",                           2009, "mainline", 8),
    (8,  "Resident Evil: Revelations",                2012, "spinoff",  7),
    (9,  "Resident Evil 6",                           2012, "mainline", 10),
    (10, "Resident Evil: Revelations 2",              2015, "spinoff",  9),
    (11, "Resident Evil 7: Biohazard",                2017, "mainline", 11),
    (12, "Resident Evil Village",                     2021, "mainline", 12),
    (13, "Resident Evil Village – Shadow of Rose",    2022, "spinoff",  13),
    (14, "Resident Evil Requiem",                     2026, "mainline", 14),
]

games = pd.DataFrame(
    games_data,
    columns=["id", "title", "year", "type", "chronology_order"]
)

games.to_csv("games.csv", index=False)
games.head(20)

,id,title,year,type,chronology_order
0,1,Resident Evil,1996,mainline,2
1,2,Resident Evil 2,1998,mainline,4
2,3,Resident Evil 3: Nemesis,1999,mainline,3
3,4,Resident Evil: Code – Veronica,2000,mainline,5
4,5,Resident Evil 0,2002,mainline,1
5,6,Resident Evil 4,2005,mainline,6
6,7,Resident Evil 5,2009,mainline,8
7,8,Resident Evil: Revelations,2012,spinoff,7
8,9,Resident Evil 6,2012,mainline,10
9,10,Resident Evil: Revelations 2,2015,spinoff,9


## Characters and GameAppearence

In [3]:
characters = pd.DataFrame(
    columns=[
        "id",       
        "name",    
        "role",     
        #"faction",  
    ]
)

gameAppearance = pd.DataFrame(
    columns=[
        "gameId",       
        "characterId", 
        "role",        
    ]
)

In [4]:
# Resident evil 4 (2005): easy scraping from wikipedia
game_id = 6
URL = "https://it.wikipedia.org/wiki/Resident_Evil_4"
headers = {
    "User-Agent": (
        "Mozilla/5.0 (X11; Linux x86_64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/120.0 Safari/537.36"
    )
}

resp = requests.get(URL, headers=headers)
soup = BeautifulSoup(resp.text, "html.parser")

names = []
roles = [] 
current_role = None 

for tag in soup.find_all(["h2", "h3"]):
    if tag.name == "h2":
        text = tag.get_text(strip=True)
        print("H2:", text)  

        if text == "Personaggi":
            current_role = "hero"
        elif text == "Nemici":
            current_role = "villain"
        else:
            current_role = None

    elif tag.name == "h3" and current_role is not None:
        text = tag.get_text(strip=True)

        names.append(text)
        roles.append(current_role)
        print("  H3:", text)

print("\n# characters:", len(names))
print(list(zip(names, roles)))


re4_df = pd.DataFrame({
    "name": names,
    "role_in_game": roles,  
})

# Characters
re4_df.loc[re4_df["name"] == "Albert Wesker", "role_in_game"] = "villain"
existing_names = set(characters["name"])

missing_mask = ~re4_df["name"].isin(existing_names)
missing_rows = re4_df[missing_mask]

next_id = characters["id"].max() + 1 if len(characters) > 0 else 1

new_rows = []
for _, row in missing_rows.iterrows():
    new_rows.append({
        "id": next_id,
        "name": row["name"],
        "role": row["role_in_game"],
    })
    next_id += 1

if new_rows:
    new_chars = pd.DataFrame(new_rows)
    characters = pd.concat([characters, new_chars], ignore_index=True)

# Game appearences
rows = []
for _, row in re4_df.iterrows():
    name = row["name"]
    role_in_game = row["role_in_game"]

    match = characters[characters["name"] == name]
    if match.empty:
        continue
    cid = match["id"].iloc[0]

    rows.append({
        "gameId": game_id,
        "characterId": cid,
        "role": role_in_game,
    })

gameApp_re4 = pd.DataFrame(rows)

gameAppearance = pd.concat([gameAppearance, gameApp_re4], ignore_index=True)

print(f"\n\ncharacters: {characters}")
print(f"\n\nGame appearence: {gameAppearance}")


# characters: 0
[]


characters: Empty DataFrame
Columns: [id, name, role]
Index: []


Game appearence: Empty DataFrame
Columns: [gameId, characterId, role]
Index: []


In [5]:
# Resident evil (1996): AI generated
game_id = 1 
next_id = characters["id"].max() + 1 

character_data = [
    # name,                 main_alignment
    ("Chris Redfield",      "hero"),
    ("Jill Valentine",      "hero"),
    ("Albert Wesker",       "villain"),
    ("Barry Burton",        "hero"),
    ("Brad Vickers",        "support"),
    ("Joseph Frost",        "support"),
    ("Rebecca Chambers",    "support"),
    ("Richard Aiken",       "support"),
    ("Enrico Marini",       "support"),
    ("Forest Speyer",       "support"),
    ("Kenneth J. Sullivan", "support"),
]
existing_names = set(characters["name"])

new_rows = []
for name, role in character_data:
    if name in existing_names:           
        continue
    new_rows.append({
        "id": next_id,
        "name": name,
        "role": role, 
    })
    next_id += 1

new_chars_df = pd.DataFrame(new_rows, columns=["id", "name", "role"])

characters = pd.concat([characters, new_chars_df], ignore_index=True)

# Game appearence
name_to_id = dict(zip(characters["name"], characters["id"]))
gameAppearance_data = [
    # name,                 role_in_game
    ("Chris Redfield",      "hero"),
    ("Jill Valentine",      "hero"),
    ("Albert Wesker",       "villain"),
    ("Barry Burton",        "support"),
    ("Brad Vickers",        "support"),
    ("Joseph Frost",        "support"),
    ("Rebecca Chambers",    "support"),
    ("Richard Aiken",       "support"),
    ("Enrico Marini",       "support"),
    ("Forest Speyer",       "support"),
    ("Kenneth J. Sullivan", "support"),
]

rows = []
for name, role_in_game in gameAppearance_data:
    cid = name_to_id[name]
    rows.append({
        "gameId": game_id,
        "characterId": cid,
        "role": role_in_game,
    })

gameApp_re1 = pd.DataFrame(rows, columns=["gameId", "characterId", "role"])
gameAppearance = pd.concat([gameAppearance, gameApp_re1], ignore_index=True)


print(f"\n\ncharacters: {characters}")
print(f"\n\nGame appearence: {gameAppearance}")



characters:      id                 name     role
0   NaN       Chris Redfield     hero
1   NaN       Jill Valentine     hero
2   NaN        Albert Wesker  villain
3   NaN         Barry Burton     hero
4   NaN         Brad Vickers  support
5   NaN         Joseph Frost  support
6   NaN     Rebecca Chambers  support
7   NaN        Richard Aiken  support
8   NaN        Enrico Marini  support
9   NaN        Forest Speyer  support
10  NaN  Kenneth J. Sullivan  support


Game appearence:    gameId characterId     role
0       1         NaN     hero
1       1         NaN     hero
2       1         NaN  villain
3       1         NaN  support
4       1         NaN  support
5       1         NaN  support
6       1         NaN  support
7       1         NaN  support
8       1         NaN  support
9       1         NaN  support
10      1         NaN  support


In [6]:
# Resident evil 2 (1998): AI generated
game_id = 2
next_id = characters["id"].max() + 1 

character_data = [
    # name,              main_alignment
    ("Leon Scott Kennedy", "hero"),      
    ("Claire Redfield",    "hero"),
    ("Ada Wong",           "hero"),      
    ("Sherry Birkin",      "support"),
    ("William Birkin",     "villain"),
    ("Annette Birkin",     "support"),
    ("Brian Irons",        "villain"),
    ("Marvin Branagh",     "support"),
    ("Ben Bertolucci",     "support"),
    ("Robert Kendo",       "support"),
]

existing_names = set(characters["name"])
new_rows = []

for name, role in character_data:
    if name in existing_names:
        continue 
    new_rows.append({
        "id": next_id,
        "name": name,
        "role": role,
    })
    next_id += 1

if new_rows:
    new_chars_df = pd.DataFrame(new_rows, columns=["id", "name", "role"])
    characters = pd.concat([characters, new_chars_df], ignore_index=True)

# Game appearence
name_to_id = dict(zip(characters["name"], characters["id"]))

gameAppearance_data = [
    # name,              role_in_game
    ("Leon Scott Kennedy", "hero"),       
    ("Claire Redfield",    "hero"),        
    ("Ada Wong",           "support"),     
    ("Sherry Birkin",      "support"),
    ("William Birkin",     "villain"),
    ("Annette Birkin",     "support"),
    ("Brian Irons",        "villain"),
    ("Marvin Branagh",     "support"),
    ("Ben Bertolucci",     "support"),
    ("Robert Kendo",       "support"),
]

rows = []
for name, role_in_game in gameAppearance_data:
    cid = name_to_id[name]
    rows.append({
        "gameId": game_id,
        "characterId": cid,
        "role": role_in_game,
    })

gameApp_re2 = pd.DataFrame(rows, columns=["gameId", "characterId", "role"])
gameAppearance = pd.concat([gameAppearance, gameApp_re2], ignore_index=True)
print(characters[characters["name"].isin([n for n, _ in character_data])])
print(gameAppearance[gameAppearance["gameId"] == 2])

     id                name     role
11  NaN  Leon Scott Kennedy     hero
12  NaN     Claire Redfield     hero
13  NaN            Ada Wong     hero
14  NaN       Sherry Birkin  support
15  NaN      William Birkin  villain
16  NaN      Annette Birkin  support
17  NaN         Brian Irons  villain
18  NaN      Marvin Branagh  support
19  NaN      Ben Bertolucci  support
20  NaN        Robert Kendo  support
   gameId characterId     role
11      2         NaN     hero
12      2         NaN     hero
13      2         NaN  support
14      2         NaN  support
15      2         NaN  villain
16      2         NaN  support
17      2         NaN  villain
18      2         NaN  support
19      2         NaN  support
20      2         NaN  support


In [7]:
# Resident evil 3: Nemesis (1999): AI generated
game_id = 3
next_id = characters["id"].max() + 1 

character_data = [
    # name,              main_alignment
    ("Jill Valentine",   "hero"),      
    ("Carlos Oliveira",  "hero"),
    ("Nemesis",          "villain"),
    ("Nikolai Zinoviev", "villain"),
    ("Mikhail Victor",   "support"),
    ("Brad Vickers",     "support"),   
    ("Marvin Branagh",   "support"),
    ("Dario Rosso",      "support"),
]

existing_names = set(characters["name"])
new_rows = []

for name, role in character_data:
    if name in existing_names:
        continue
    new_rows.append({
        "id": next_id,
        "name": name,
        "role": role,
    })
    next_id += 1

if new_rows:
    new_chars_df = pd.DataFrame(new_rows, columns=["id", "name", "role"])
    characters = pd.concat([characters, new_chars_df], ignore_index=True)

# Game appearence
name_to_id = dict(zip(characters["name"], characters["id"]))

gameAppearance_data = [
    # name,              role_in_game
    ("Jill Valentine",   "hero"),       
    ("Carlos Oliveira",  "hero"),       
    ("Nemesis",          "villain"), 
    ("Nikolai Zinoviev", "villain"),
    ("Mikhail Victor",   "support"),
    ("Brad Vickers",     "support"),
    ("Marvin Branagh",   "support"),
    ("Dario Rosso",      "support"),
]

rows = []
for name, role_in_game in gameAppearance_data:
    cid = name_to_id[name]
    rows.append({
        "gameId": game_id,
        "characterId": cid,
        "role": role_in_game,
    })

gameApp_re3 = pd.DataFrame(rows, columns=["gameId", "characterId", "role"])
gameAppearance = pd.concat([gameAppearance, gameApp_re3], ignore_index=True)

print(characters[characters["name"].isin([n for n, _ in character_data])])
print(gameAppearance[gameAppearance["gameId"] == 3])

     id              name     role
1   NaN    Jill Valentine     hero
4   NaN      Brad Vickers  support
18  NaN    Marvin Branagh  support
21  NaN   Carlos Oliveira     hero
22  NaN           Nemesis  villain
23  NaN  Nikolai Zinoviev  villain
24  NaN    Mikhail Victor  support
25  NaN       Dario Rosso  support
   gameId characterId     role
21      3         NaN     hero
22      3         NaN     hero
23      3         NaN  villain
24      3         NaN  villain
25      3         NaN  support
26      3         NaN  support
27      3         NaN  support
28      3         NaN  support


In [8]:
# Resident evil: Code – Veronica (2000): AI generated
game_id = 4
next_id = characters["id"].max() + 1 

character_data = [
    # name,              main_alignment
    ("Claire Redfield",  "hero"),
    ("Steve Burnside",   "hero"),
    ("Chris Redfield",   "hero"),
    ("Albert Wesker",    "villain"),
    ("Alexia Ashford",   "villain"),
    ("Alfred Ashford",   "villain"),
    ("Rodrigo Juan Raval","support"),
]

existing_names = set(characters["name"])
new_rows = []

for name, role in character_data:
    if name in existing_names:
        continue
    new_rows.append({
        "id": next_id,
        "name": name,
        "role": role,
    })
    next_id += 1

if new_rows:
    new_chars_df = pd.DataFrame(new_rows, columns=["id", "name", "role"])
    characters = pd.concat([characters, new_chars_df], ignore_index=True)

# Game appearence
name_to_id = dict(zip(characters["name"], characters["id"]))

gameAppearance_data = [
    # name,              role_in_game
    ("Claire Redfield",  "hero"),        
    ("Steve Burnside",   "hero"),       
    ("Chris Redfield",   "hero"),       
    ("Albert Wesker",    "villain"), 
    ("Alexia Ashford",   "villain"), 
    ("Alfred Ashford",   "villain"),
    ("Rodrigo Juan Raval","support"),
]

rows = []
for name, role_in_game in gameAppearance_data:
    cid = name_to_id[name]
    rows.append({
        "gameId": game_id,
        "characterId": cid,
        "role": role_in_game,
    })

gameApp_cvx = pd.DataFrame(rows, columns=["gameId", "characterId", "role"])
gameAppearance = pd.concat([gameAppearance, gameApp_cvx], ignore_index=True)

print(characters[characters["name"].isin([n for n, _ in character_data])])
print(gameAppearance[gameAppearance["gameId"] == game_id])

     id                name     role
0   NaN      Chris Redfield     hero
2   NaN       Albert Wesker  villain
12  NaN     Claire Redfield     hero
26  NaN      Steve Burnside     hero
27  NaN      Alexia Ashford  villain
28  NaN      Alfred Ashford  villain
29  NaN  Rodrigo Juan Raval  support
   gameId characterId     role
29      4         NaN     hero
30      4         NaN     hero
31      4         NaN     hero
32      4         NaN  villain
33      4         NaN  villain
34      4         NaN  villain
35      4         NaN  support


In [9]:
# Resident evil 0 (2002): AI generated
game_id = 5
next_id = characters["id"].max() + 1 

character_data = [
    # name,              main_alignment
    ("Rebecca Chambers", "hero"),
    ("Billy Coen",       "hero"),
    ("James Marcus",     "villain"),
    ("Edward Dewey",     "support"),
    ("Enrico Marini",    "support"),
    ("Albert Wesker",    "villain"),
    ("William Birkin",   "villain"),
]

existing_names = set(characters["name"])
new_rows = []

for name, role in character_data:
    if name in existing_names:
        continue
    new_rows.append({
        "id": next_id,
        "name": name,
        "role": role,
    })
    next_id += 1

if new_rows:
    new_chars_df = pd.DataFrame(new_rows, columns=["id", "name", "role"])
    characters = pd.concat([characters, new_chars_df], ignore_index=True)

# Game appearence
name_to_id = dict(zip(characters["name"], characters["id"]))

gameAppearance_data = [
    # name,              role_in_game
    ("Rebecca Chambers", "hero"),       
    ("Billy Coen",       "hero"),        
    ("James Marcus",     "villain"),  
    ("Edward Dewey",     "support"),
    ("Enrico Marini",    "support"),
    ("Albert Wesker",    "villain"),
    ("William Birkin",   "villain"),
]

rows = []
for name, role_in_game in gameAppearance_data:
    cid = name_to_id[name]
    rows.append({
        "gameId": game_id,
        "characterId": cid,
        "role": role_in_game,
    })

gameApp_re0 = pd.DataFrame(rows, columns=["gameId", "characterId", "role"])
gameAppearance = pd.concat([gameAppearance, gameApp_re0], ignore_index=True)

print(characters[characters["name"].isin([n for n, _ in character_data])])
print(gameAppearance[gameAppearance["gameId"] == game_id])


     id              name     role
2   NaN     Albert Wesker  villain
6   NaN  Rebecca Chambers  support
8   NaN     Enrico Marini  support
15  NaN    William Birkin  villain
30  NaN        Billy Coen     hero
31  NaN      James Marcus  villain
32  NaN      Edward Dewey  support
   gameId characterId     role
36      5         NaN     hero
37      5         NaN     hero
38      5         NaN  villain
39      5         NaN  support
40      5         NaN  support
41      5         NaN  villain
42      5         NaN  villain


In [10]:
# Resident evil 5 (2009): AI generated
game_id = 7
next_id = characters["id"].max() + 1 

character_data = [
    # name,              main_alignment
    ("Chris Redfield",  "hero"),
    ("Sheva Alomar",    "hero"),
    ("Albert Wesker",   "villain"),
    ("Jill Valentine",  "hero"),
    ("Ricardo Irving",  "villain"),
    ("Excella Gionne",  "villain"),
    ("Josh Stone",      "support"),
    ("Ozwell E. Spencer","villain"),
]

existing_names = set(characters["name"])
new_rows = []

for name, role in character_data:
    if name in existing_names:
        continue
    new_rows.append({
        "id": next_id,
        "name": name,
        "role": role,
    })
    next_id += 1

if new_rows:
    new_chars_df = pd.DataFrame(new_rows, columns=["id", "name", "role"])
    characters = pd.concat([characters, new_chars_df], ignore_index=True)

# Game appearence
name_to_id = dict(zip(characters["name"], characters["id"]))

gameAppearance_data = [
    # name,              role_in_game
    ("Chris Redfield",   "hero"),        
    ("Sheva Alomar",     "hero"),        
    ("Albert Wesker",    "villain"),  
    ("Jill Valentine",   "support"),     
    ("Ricardo Irving",   "villain"),
    ("Excella Gionne",   "villain"),
    ("Josh Stone",       "support"),
    ("Ozwell E. Spencer","support"),     
]

rows = []
for name, role_in_game in gameAppearance_data:
    cid = name_to_id[name]
    rows.append({
        "gameId": game_id,
        "characterId": cid,
        "role": role_in_game,
    })

gameApp_re5 = pd.DataFrame(rows, columns=["gameId", "characterId", "role"])
gameAppearance = pd.concat([gameAppearance, gameApp_re5], ignore_index=True)

print(characters[characters["name"].isin([n for n, _ in character_data])])
print(gameAppearance[gameAppearance["gameId"] == game_id])

     id               name     role
0   NaN     Chris Redfield     hero
1   NaN     Jill Valentine     hero
2   NaN      Albert Wesker  villain
33  NaN       Sheva Alomar     hero
34  NaN     Ricardo Irving  villain
35  NaN     Excella Gionne  villain
36  NaN         Josh Stone  support
37  NaN  Ozwell E. Spencer  villain
   gameId characterId     role
43      7         NaN     hero
44      7         NaN     hero
45      7         NaN  villain
46      7         NaN  support
47      7         NaN  villain
48      7         NaN  villain
49      7         NaN  support
50      7         NaN  support


In [11]:
# Resident Evil: Revelations (2012): AI generated
game_id = 8
next_id = characters["id"].max() + 1 

character_data = [
    # name,              main_alignment
    ("Jill Valentine",   "hero"),
    ("Chris Redfield",   "hero"),
    ("Parker Luciani",   "hero"),
    ("Jessica Sherawat", "hero"),    
    ("Clive R. O'Brian", "support"),
    ("Raymond Vester",   "support"),  
    ("Morgan Lansdale",  "villain"),
    ("Jack Norman",      "villain"),
    ("Rachel Foley",     "support"),
    ("Keith Lumley",     "support"),
    ("Quint Cetcham",    "support"),
]

existing_names = set(characters["name"])
new_rows = []

for name, role in character_data:
    if name in existing_names:
        continue
    new_rows.append({
        "id": next_id,
        "name": name,
        "role": role,
    })
    next_id += 1

if new_rows:
    new_chars_df = pd.DataFrame(new_rows, columns=["id", "name", "role"])
    characters = pd.concat([characters, new_chars_df], ignore_index=True)

# Game appearence
name_to_id = dict(zip(characters["name"], characters["id"]))

gameAppearance_data = [
    # name,              role_in_game
    ("Jill Valentine",   "hero"),
    ("Chris Redfield",   "hero"),
    ("Parker Luciani",   "support"),
    ("Jessica Sherawat", "support"),
    ("Clive R. O'Brian", "support"),
    ("Raymond Vester",   "support"),    
    ("Morgan Lansdale",  "villain"), 
    ("Jack Norman",      "villain"),
    ("Rachel Foley",     "support"),
    ("Keith Lumley",     "support"),
    ("Quint Cetcham",    "support"),
]

rows = []
for name, role_in_game in gameAppearance_data:
    cid = name_to_id[name]
    rows.append({
        "gameId": game_id,
        "characterId": cid,
        "role": role_in_game,
    })

gameApp_rev = pd.DataFrame(rows, columns=["gameId", "characterId", "role"])
gameAppearance = pd.concat([gameAppearance, gameApp_rev], ignore_index=True)

print(characters[characters["name"].isin([n for n, _ in character_data])])
print(gameAppearance[gameAppearance["gameId"] == game_id])

     id              name     role
0   NaN    Chris Redfield     hero
1   NaN    Jill Valentine     hero
38  NaN    Parker Luciani     hero
39  NaN  Jessica Sherawat     hero
40  NaN  Clive R. O'Brian  support
41  NaN    Raymond Vester  support
42  NaN   Morgan Lansdale  villain
43  NaN       Jack Norman  villain
44  NaN      Rachel Foley  support
45  NaN      Keith Lumley  support
46  NaN     Quint Cetcham  support
   gameId characterId     role
51      8         NaN     hero
52      8         NaN     hero
53      8         NaN  support
54      8         NaN  support
55      8         NaN  support
56      8         NaN  support
57      8         NaN  villain
58      8         NaN  villain
59      8         NaN  support
60      8         NaN  support
61      8         NaN  support


In [12]:
# Resident evil 6 (2012): AI generated
game_id = 9
next_id = characters["id"].max() + 1 

character_data = [
    # name,              main_alignment
    ("Leon Scott Kennedy", "hero"),
    ("Chris Redfield",     "hero"),
    ("Jake Muller",        "hero"),
    ("Ada Wong",           "hero"),      
    ("Sherry Birkin",      "support"),
    ("Helena Harper",      "hero"),
    ("Piers Nivans",       "hero"),
    ("Ingrid Hunnigan",    "support"),
    ("Derek C. Simmons",   "villain"),
]

existing_names = set(characters["name"])
new_rows = []

for name, role in character_data:
    if name in existing_names:
        continue
    new_rows.append({
        "id": next_id,
        "name": name,
        "role": role,
    })
    next_id += 1

if new_rows:
    new_chars_df = pd.DataFrame(new_rows, columns=["id", "name", "role"])
    characters = pd.concat([characters, new_chars_df], ignore_index=True)

# Game appearence
name_to_id = dict(zip(characters["name"], characters["id"]))

gameAppearance_data = [
    # name,              role_in_game
    ("Leon Scott Kennedy", "hero"),        
    ("Helena Harper",      "hero"),        
    ("Chris Redfield",     "hero"),       
    ("Piers Nivans",       "hero"),        
    ("Jake Muller",        "hero"),       
    ("Sherry Birkin",      "support"),    
    ("Ada Wong",           "support"),    
    ("Ingrid Hunnigan",    "support"),
    ("Derek C. Simmons",   "villain"),
]

rows = []
for name, role_in_game in gameAppearance_data:
    cid = name_to_id[name]
    rows.append({
        "gameId": game_id,
        "characterId": cid,
        "role": role_in_game,
    })

gameApp_re6 = pd.DataFrame(rows, columns=["gameId", "characterId", "role"])
gameAppearance = pd.concat([gameAppearance, gameApp_re6], ignore_index=True)

print(characters[characters["name"].isin([n for n, _ in character_data])])
print(gameAppearance[gameAppearance["gameId"] == game_id])

     id                name     role
0   NaN      Chris Redfield     hero
11  NaN  Leon Scott Kennedy     hero
13  NaN            Ada Wong     hero
14  NaN       Sherry Birkin  support
47  NaN         Jake Muller     hero
48  NaN       Helena Harper     hero
49  NaN        Piers Nivans     hero
50  NaN     Ingrid Hunnigan  support
51  NaN    Derek C. Simmons  villain
   gameId characterId     role
62      9         NaN     hero
63      9         NaN     hero
64      9         NaN     hero
65      9         NaN     hero
66      9         NaN     hero
67      9         NaN  support
68      9         NaN  support
69      9         NaN  support
70      9         NaN  villain


In [13]:
# Resident Evil: Revelations 2 (2015): AI generated
game_id = 10
next_id = characters["id"].max() + 1 

character_data = [
    # name,            main_alignment
    ("Claire Redfield", "hero"),
    ("Barry Burton",    "hero"),
    ("Moira Burton",    "support"),
    ("Natalia Korda",   "support"),
    ("Alex Wesker",     "villain"),
]

existing_names = set(characters["name"])
new_rows = []

for name, role in character_data:
    if name in existing_names:
        continue
    new_rows.append({
        "id": next_id,
        "name": name,
        "role": role,
    })
    next_id += 1

if new_rows:
    new_chars_df = pd.DataFrame(new_rows, columns=["id", "name", "role"])
    characters = pd.concat([characters, new_chars_df], ignore_index=True)

# Game appearence
name_to_id = dict(zip(characters["name"], characters["id"]))

gameAppearance_data = [
    # name,            role_in_game
    ("Claire Redfield", "hero"),
    ("Barry Burton",    "hero"),
    ("Moira Burton",    "support"),
    ("Natalia Korda",   "support"),
    ("Alex Wesker",     "villain"),
]

rows = []
for name, role_in_game in gameAppearance_data:
    cid = name_to_id[name]
    rows.append({
        "gameId": game_id,
        "characterId": cid,
        "role": role_in_game,
    })

gameApp_rev2 = pd.DataFrame(rows, columns=["gameId", "characterId", "role"])
gameAppearance = pd.concat([gameAppearance, gameApp_rev2], ignore_index=True)

print(characters[characters["name"].isin([n for n, _ in character_data])])
print(gameAppearance[gameAppearance["gameId"] == game_id])

     id             name     role
3   NaN     Barry Burton     hero
12  NaN  Claire Redfield     hero
52  NaN     Moira Burton  support
53  NaN    Natalia Korda  support
54  NaN      Alex Wesker  villain
   gameId characterId     role
71     10         NaN     hero
72     10         NaN     hero
73     10         NaN  support
74     10         NaN  support
75     10         NaN  villain


In [14]:
# Resident Evil 7: Biohazard (2017): AI generated
game_id = 11
next_id = characters["id"].max() + 1 

character_data = [
    # name,           main_alignment
    ("Ethan Winters", "hero"),
    ("Mia Winters",   "hero"),
    ("Jack Baker",    "villain"),
    ("Marguerite Baker","villain"),
    ("Lucas Baker",   "villain"),
    ("Zoe Baker",     "support"),
    ("Eveline",       "villain"),
]

existing_names = set(characters["name"])
new_rows = []

for name, role in character_data:
    if name in existing_names:
        continue
    new_rows.append({
        "id": next_id,
        "name": name,
        "role": role,
    })
    next_id += 1

if new_rows:
    new_chars_df = pd.DataFrame(new_rows, columns=["id", "name", "role"])
    characters = pd.concat([characters, new_chars_df], ignore_index=True)

# Game appearence
name_to_id = dict(zip(characters["name"], characters["id"]))

gameAppearance_data = [
    # name,           role_in_game
    ("Ethan Winters", "hero"),
    ("Mia Winters",   "support"),     
    ("Jack Baker",    "villain"),
    ("Marguerite Baker","villain"),
    ("Lucas Baker",   "villain"),
    ("Zoe Baker",     "support"),
    ("Eveline",       "villain"),
]

rows = []
for name, role_in_game in gameAppearance_data:
    cid = name_to_id[name]
    rows.append({
        "gameId": game_id,
        "characterId": cid,
        "role": role_in_game,
    })

gameApp_re7 = pd.DataFrame(rows, columns=["gameId", "characterId", "role"])
gameAppearance = pd.concat([gameAppearance, gameApp_re7], ignore_index=True)

print(characters[characters["name"].isin([n for n, _ in character_data])])
print(gameAppearance[gameAppearance["gameId"] == game_id])

     id              name     role
55  NaN     Ethan Winters     hero
56  NaN       Mia Winters     hero
57  NaN        Jack Baker  villain
58  NaN  Marguerite Baker  villain
59  NaN       Lucas Baker  villain
60  NaN         Zoe Baker  support
61  NaN           Eveline  villain
   gameId characterId     role
76     11         NaN     hero
77     11         NaN  support
78     11         NaN  villain
79     11         NaN  villain
80     11         NaN  villain
81     11         NaN  support
82     11         NaN  villain


In [15]:
# Resident Evil Village (2021): AI generated
game_id = 12
next_id = characters["id"].max() + 1 

character_data = [
    # name,               main_alignment
    ("Ethan Winters",     "hero"),
    ("Mia Winters",       "hero"),
    ("Rosemary Winters",  "support"),
    ("Chris Redfield",    "hero"),
    ("Mother Miranda",    "villain"),
    ("Alcina Dimitrescu", "villain"),
    ("Karl Heisenberg",   "villain"),
    ("Salvatore Moreau",  "villain"),
    ("Donna Beneviento",  "villain"),
]

existing_names = set(characters["name"])
new_rows = []

for name, role in character_data:
    if name in existing_names:
        continue
    new_rows.append({
        "id": next_id,
        "name": name,
        "role": role,
    })
    next_id += 1

if new_rows:
    new_chars_df = pd.DataFrame(new_rows, columns=["id", "name", "role"])
    characters = pd.concat([characters, new_chars_df], ignore_index=True)

# Game appearence
name_to_id = dict(zip(characters["name"], characters["id"]))

gameAppearance_data = [
    # name,               role_in_game
    ("Ethan Winters",     "hero"),
    ("Mia Winters",       "support"),
    ("Rosemary Winters",  "support"),
    ("Chris Redfield",    "support"),
    ("Mother Miranda",    "villain"),
    ("Alcina Dimitrescu", "villain"),
    ("Karl Heisenberg",   "villain"),
    ("Salvatore Moreau",  "villain"),
    ("Donna Beneviento",  "villain"),
]

rows = []
for name, role_in_game in gameAppearance_data:
    cid = name_to_id[name]
    rows.append({
        "gameId": game_id,
        "characterId": cid,
        "role": role_in_game,
    })

gameApp_village = pd.DataFrame(rows, columns=["gameId", "characterId", "role"])
gameAppearance = pd.concat([gameAppearance, gameApp_village], ignore_index=True)

print(characters[characters["name"].isin([n for n, _ in character_data])])
print(gameAppearance[gameAppearance["gameId"] == game_id])

     id               name     role
0   NaN     Chris Redfield     hero
55  NaN      Ethan Winters     hero
56  NaN        Mia Winters     hero
62  NaN   Rosemary Winters  support
63  NaN     Mother Miranda  villain
64  NaN  Alcina Dimitrescu  villain
65  NaN    Karl Heisenberg  villain
66  NaN   Salvatore Moreau  villain
67  NaN   Donna Beneviento  villain
   gameId characterId     role
83     12         NaN     hero
84     12         NaN  support
85     12         NaN  support
86     12         NaN  support
87     12         NaN  villain
88     12         NaN  villain
89     12         NaN  villain
90     12         NaN  villain
91     12         NaN  villain


In [16]:
# Resident Evil Village – Shadow of Rose (2022): AI generated
game_id = 13
next_id = characters["id"].max() + 1 

character_data = [
    # name,              main_alignment
    ("Rosemary Winters", "hero"),
    ("Ethan Winters",    "hero"),
    ("Mother Miranda",   "villain"),
]

existing_names = set(characters["name"])
new_rows = []

for name, role in character_data:
    if name in existing_names:
        continue
    new_rows.append({
        "id": next_id,
        "name": name,
        "role": role,
    })
    next_id += 1

if new_rows:
    new_chars_df = pd.DataFrame(new_rows, columns=["id", "name", "role"])
    characters = pd.concat([characters, new_chars_df], ignore_index=True)

# Game appearence
name_to_id = dict(zip(characters["name"], characters["id"]))

gameAppearance_data = [
    # name,              role_in_game
    ("Rosemary Winters", "hero"),
    ("Ethan Winters",    "support"),
    ("Mother Miranda",   "villain"),
]

rows = []
for name, role_in_game in gameAppearance_data:
    cid = name_to_id[name]
    rows.append({
        "gameId": game_id,
        "characterId": cid,
        "role": role_in_game,
    })

gameApp_sor = pd.DataFrame(rows, columns=["gameId", "characterId", "role"])
gameAppearance = pd.concat([gameAppearance, gameApp_sor], ignore_index=True)

print(characters[characters["name"].isin([n for n, _ in character_data])])
print(gameAppearance[gameAppearance["gameId"] == game_id])

     id              name     role
55  NaN     Ethan Winters     hero
62  NaN  Rosemary Winters  support
63  NaN    Mother Miranda  villain
   gameId characterId     role
92     13         NaN     hero
93     13         NaN  support
94     13         NaN  villain


In [17]:
# Resident Evil Requiem (2026): AI generated
game_id = 14
next_id = characters["id"].max() + 1 

character_data = [
    # name,              main_alignment
    ("Grace Ashcroft",   "hero"),
    ("Leon Scott Kennedy","hero"),
    ("Sherry Birkin",    "support"),
    ("Victor Gideon",    "villain"),
    ("Nathan Dempsey",     "support"),
    ("Alyssa Ashcroft",   "support"),
]

existing_names = set(characters["name"])
new_rows = []

for name, role in character_data:
    if name in existing_names:
        continue
    new_rows.append({
        "id": next_id,
        "name": name,
        "role": role,
    })
    next_id += 1

if new_rows:
    new_chars_df = pd.DataFrame(new_rows, columns=["id", "name", "role"])
    characters = pd.concat([characters, new_chars_df], ignore_index=True)

# Game appearence
name_to_id = dict(zip(characters["name"], characters["id"]))

gameAppearance_data = [
    # name,              role_in_game
    ("Grace Ashcroft",    "hero"),       
    ("Leon Scott Kennedy","hero"),       
    ("Sherry Birkin",     "support"),
    ("Victor Gideon",     "villain"),
    ("Nathan Dempsey",     "support"),
    ("Alyssa Ashcroft",   "support")
]

rows = []
for name, role_in_game in gameAppearance_data:
    cid = name_to_id[name]
    rows.append({
        "gameId": game_id,
        "characterId": cid,
        "role": role_in_game,
    })

gameApp_requiem = pd.DataFrame(rows, columns=["gameId", "characterId", "role"])
gameAppearance = pd.concat([gameAppearance, gameApp_requiem], ignore_index=True)

print(characters[characters["name"].isin([n for n, _ in character_data])])
print(gameAppearance[gameAppearance["gameId"] == game_id])

     id                name     role
11  NaN  Leon Scott Kennedy     hero
14  NaN       Sherry Birkin  support
68  NaN      Grace Ashcroft     hero
69  NaN       Victor Gideon  villain
70  NaN      Nathan Dempsey  support
71  NaN     Alyssa Ashcroft  support
    gameId characterId     role
95      14         NaN     hero
96      14         NaN     hero
97      14         NaN  support
98      14         NaN  villain
99      14         NaN  support
100     14         NaN  support


In [18]:
characters.to_csv("characters.csv", index=False)
print(characters.to_string())

     id                 name     role
0   NaN       Chris Redfield     hero
1   NaN       Jill Valentine     hero
2   NaN        Albert Wesker  villain
3   NaN         Barry Burton     hero
4   NaN         Brad Vickers  support
5   NaN         Joseph Frost  support
6   NaN     Rebecca Chambers  support
7   NaN        Richard Aiken  support
8   NaN        Enrico Marini  support
9   NaN        Forest Speyer  support
10  NaN  Kenneth J. Sullivan  support
11  NaN   Leon Scott Kennedy     hero
12  NaN      Claire Redfield     hero
13  NaN             Ada Wong     hero
14  NaN        Sherry Birkin  support
15  NaN       William Birkin  villain
16  NaN       Annette Birkin  support
17  NaN          Brian Irons  villain
18  NaN       Marvin Branagh  support
19  NaN       Ben Bertolucci  support
20  NaN         Robert Kendo  support
21  NaN      Carlos Oliveira     hero
22  NaN              Nemesis  villain
23  NaN     Nikolai Zinoviev  villain
24  NaN       Mikhail Victor  support
25  NaN     

In [19]:
gameAppearance = (
    gameAppearance
    .sort_values(by=["gameId", "characterId"])
    .reset_index(drop=True)
)

gameAppearance.to_csv("gameAppearance.csv", index=False)
print(gameAppearance.to_string())

    gameId characterId     role
0        1         NaN     hero
1        1         NaN     hero
2        1         NaN  villain
3        1         NaN  support
4        1         NaN  support
5        1         NaN  support
6        1         NaN  support
7        1         NaN  support
8        1         NaN  support
9        1         NaN  support
10       1         NaN  support
11       2         NaN     hero
12       2         NaN     hero
13       2         NaN  support
14       2         NaN  support
15       2         NaN  villain
16       2         NaN  support
17       2         NaN  villain
18       2         NaN  support
19       2         NaN  support
20       2         NaN  support
21       3         NaN     hero
22       3         NaN     hero
23       3         NaN  villain
24       3         NaN  villain
25       3         NaN  support
26       3         NaN  support
27       3         NaN  support
28       3         NaN  support
29       4         NaN     hero
30      